# Name That LLM: 4 visões + LR, atribuição em grupos com teto

Gerado por `scripts/build_kaggle_notebook.py` a partir de [kaggle-name-that-llm](https://github.com/pedrohbenetton/kaggle-name-that-llm). Validação: GroupKFold por prompt (balanced accuracy). As visões de estilo (forma, palavras funcionais) e o agrupamento com teto seguem ideias dos notebooks públicos da competição; a implementação é própria.

In [ ]:
# ---- ntl/data.py ----
"""Rótulos oficiais, caminhos e validação dos dados da competição."""


import os
from pathlib import Path

import pandas as pd

LABELS: tuple[str, ...] = (
    "gpt-5.5", "gpt-5.6-luna", "gpt-5.6-sol", "gpt-5.6-terra", "gpt-6-astra",
    "gemini-3.1-pro-preview", "gemini-3.6-flash", "gemini-3.7-flash", "gemini-3.8-flash",
    "claude-fable-5", "claude-opus-4-6", "claude-opus-4-8", "claude-opus-5", "claude-sonnet-5",
    "qwen3.8-max", "glm-5.3", "glm-5.3-flash", "kimi-k3",
)

# No Kaggle: /kaggle/input/... Local: pasta baixada dentro do projeto (ignorada pelo git).
# Num notebook não existe __file__, por isso o caminho local é opcional.
_LOCAL = (
    str(Path(globals()["__file__"]).resolve().parents[2] / "name-that-llm" / "name-that-llm")
    if "__file__" in globals()
    else ""
)
_CANDIDATES = (
    os.environ.get("NTL_DATA", ""),
    "/kaggle/input/competitions/name-that-llm",
    "/kaggle/input/name-that-llm",
    _LOCAL,
)


def data_dir() -> Path:
    for c in _CANDIDATES:
        if c and (Path(c) / "train.csv").exists():
            return Path(c)
    raise FileNotFoundError("train.csv não encontrado; defina NTL_DATA com a pasta dos dados")


def validate_train(df: pd.DataFrame) -> None:
    falta = {"id", "prompt_id", "text", "model"} - set(df.columns)
    if falta:
        raise ValueError(f"colunas ausentes: {sorted(falta)}")
    if df["id"].duplicated().any():
        raise ValueError("id repetido")
    if df["text"].isna().any() or (df["text"].astype(str).str.strip() == "").any():
        raise ValueError("texto vazio")
    fora = set(df["model"]) - set(LABELS)
    if fora:
        raise ValueError(f"rótulos fora da lista oficial: {sorted(fora)}")


def load_train(path: Path | None = None) -> pd.DataFrame:
    df = pd.read_csv((path or data_dir()) / "train.csv")
    validate_train(df)
    return df


def load_test(path: Path | None = None) -> pd.DataFrame:
    return pd.read_csv((path or data_dir()) / "test.csv")

In [ ]:
# ---- ntl/models.py ----
"""Modelos. Baseline: TF-IDF de palavras + caracteres e regressão logística.
`four_view_lr` acrescenta duas visões de estilo que ignoram o tema: a "forma" das palavras
(maiúscula/minúscula) e as palavras funcionais. As quatro visões e seus pesos vêm do SVC4 dos
notebooks públicos da competição; a implementação aqui é própria."""


import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import FeatureUnion, Pipeline


def tfidf_lr(max_word: int = 25000, max_char: int = 35000, C: float = 2.0) -> Pipeline:
    feats = FeatureUnion([
        ("word", TfidfVectorizer(ngram_range=(1, 2), max_features=max_word, sublinear_tf=True)),
        ("char", TfidfVectorizer(analyzer="char", ngram_range=(3, 5), max_features=max_char,
                                 sublinear_tf=True)),
    ])
    return Pipeline([("feats", feats), ("clf", LogisticRegression(C=C, max_iter=2000))])


FUNCTION_WORDS = frozenset(
    ["a", "an", "the", "and", "or", "but", "if", "while", "although", "because", "as", "so", "than", "that", "which", "who", "whom", "whose", "this", "these", "those", "it", "its", "they", "them", "their", "theirs", "he", "him", "his", "she", "her", "hers", "we", "us", "our", "ours", "you", "your", "yours", "i", "me", "my", "mine", "in", "on", "at", "to", "from", "with", "without", "for", "of", "by", "over", "under", "between", "through", "into", "about", "is", "are", "was", "were", "be", "been", "being", "have", "has", "had", "do", "does", "did", "can", "could", "will", "would", "shall", "should", "may", "might", "must", "not", "no", "nor", "all", "any", "some", "each", "every", "both", "either", "neither", "other", "another", "such", "only", "also", "just", "very", "too", "more", "most", "less", "least", "here", "there", "when", "where", "why", "how", "what", "whether", "then", "now", "still", "yet", "even", "much", "many", "few", "little"]
)
_WORD = re.compile(r"[^\W\d_]+")
_TOKEN = re.compile(r"\w+|[^\w\s]")


def _shape(w: str) -> str:
    head = "X" if w[0].isupper() else "x"
    if len(w) == 1:
        return head
    return head + ("X" if w[1:].isupper() else "x")


def shape_view(text: str) -> str:
    """Troca cada palavra pelo seu formato (Xx, x, XX); mantém pontuação, números e espaços."""
    return _WORD.sub(lambda m: _shape(m[0]), text)


def function_view(text: str) -> str:
    """Mantém palavras funcionais e pontuação; o resto vira CONTENT."""
    return " ".join(
        t.lower() if t.lower() in FUNCTION_WORDS or not t.isalnum() else "CONTENT"
        for t in _TOKEN.findall(text)
    )


VIEWS = (  # nome, analisador, n-gramas, máx. de features, peso
    ("char", "char", (3, 5), 250000, 1.0),
    ("word", "word", (1, 3), 180000, 0.7),
    ("shape", "char", (2, 6), 60000, 0.5),
    ("function", "word", (1, 3), 50000, 0.5),
)
_PRE = {"shape": shape_view, "function": function_view}


def four_view_features(max_features: int | None = None, min_df: int = 2) -> FeatureUnion:
    parts, weights = [], {}
    for name, analyzer, ngrams, cap, w in VIEWS:
        opts = {"analyzer": analyzer, "ngram_range": ngrams, "lowercase": False,
                "sublinear_tf": True, "min_df": min_df, "max_features": max_features or cap,
                "preprocessor": _PRE.get(name)}
        if analyzer == "word":
            opts["token_pattern"] = r"(?u)\b\w+\b|[^\w\s]"
        parts.append((name, TfidfVectorizer(**opts)))
        weights[name] = w
    return FeatureUnion(parts, transformer_weights=weights)


def four_view_lr(max_features: int | None = None, C: float = 4.0, min_df: int = 2) -> Pipeline:
    return Pipeline([("feats", four_view_features(max_features, min_df)),
                     ("clf", LogisticRegression(C=C, max_iter=3000))])

In [ ]:
# ---- ntl/groups.py ----
"""Infere os grupos de prompt só pelo conteúdo: textos que respondem ao mesmo prompt
compartilham vocabulário de tema. Usa TF-IDF de palavras (sem stopwords) e agrupamento
hierárquico por cosseno."""


from collections.abc import Sequence

import numpy as np
from sklearn.cluster import AgglomerativeClustering
from sklearn.feature_extraction.text import TfidfVectorizer


def infer_groups(
    texts: Sequence[str], n_groups: int | None = None, per_group: int = 18
) -> np.ndarray:
    if len(texts) < 2:
        return np.zeros(len(texts), dtype=int)
    if n_groups is None:
        n_groups = max(1, round(len(texts) / per_group))
    X = TfidfVectorizer(stop_words="english", sublinear_tf=True, min_df=1).fit_transform(texts)
    # distância de cosseno pré-computada (n x n): evita densificar a matriz n x vocabulário
    D = np.clip(1.0 - (X @ X.T).toarray(), 0.0, 2.0)
    np.fill_diagonal(D, 0.0)
    n_groups = min(n_groups, len(texts))
    clu = AgglomerativeClustering(n_clusters=n_groups, metric="precomputed", linkage="average")
    return clu.fit_predict(D)


def capped_groups(
    texts: Sequence[str], cap: int = 18, threshold: float = 0.05, max_df: float = 1.0
) -> np.ndarray:
    """Agrupamento hierárquico (ligação média) que nunca forma grupo maior que `cap` e para
    quando a melhor similaridade média cai abaixo de `threshold`. Diferente de `infer_groups`,
    não supõe n/18 grupos: grupos parciais (teste desbalanceado) não são fundidos à força.
    Ideia do teto + limiar vista nos notebooks públicos da competição; implementação própria."""
    n = len(texts)
    if n < 2:
        return np.zeros(n, dtype=int)
    try:
        X = TfidfVectorizer(stop_words="english", sublinear_tf=True, max_df=max_df).fit_transform(
            texts
        )
    except ValueError:  # vocabulário vazio
        return np.arange(n)
    A = (X @ X.T).toarray().astype(np.float32)
    np.fill_diagonal(A, -np.inf)
    size = np.ones(n, dtype=np.int64)
    alive = np.ones(n, dtype=bool)
    lab = np.arange(n)

    def row_best(k: int) -> tuple[float, int]:
        ok = alive & (size + size[k] <= cap)
        ok[k] = False
        if not ok.any():
            return -np.inf, -1
        r = np.where(ok, A[k], -np.inf)
        j = int(np.argmax(r))
        return float(r[j]), j

    best = np.full(n, -np.inf, dtype=np.float64)
    arg = np.full(n, -1)
    for k in range(n):
        best[k], arg[k] = row_best(k)

    while True:
        i = int(np.argmax(best))
        if not np.isfinite(best[i]) or best[i] < threshold:
            break
        j = int(arg[i])
        si, sj = size[i], size[j]
        new = (si * A[i] + sj * A[j]) / (si + sj)
        A[i], A[:, i] = new, new
        A[j], A[:, j] = -np.inf, -np.inf
        A[i, i] = -np.inf
        size[i] += sj
        alive[j] = False
        best[j], arg[j] = -np.inf, -1
        lab[lab == j] = i
        # linhas cujo melhor parceiro sumiu ou cresceu demais: recalcula
        for k in np.flatnonzero(alive & ((arg == i) | (arg == j))):
            best[k], arg[k] = row_best(int(k))
        best[i], arg[i] = row_best(i)
        # o grupo novo pode virar o melhor parceiro de outras linhas
        cand = alive & (size + size[i] <= cap) & (A[:, i] > best)
        cand[i] = False
        best[cand], arg[cand] = A[cand, i], i
    return np.unique(lab, return_inverse=True)[1]

In [ ]:
# ---- ntl/post.py ----
"""Pós-processamento por grupo de prompt. Cada prompt costuma ter um texto por modelo, então
(1) centralizar os scores dentro do grupo tira o viés de tema e (2) a atribuição (Hungarian)
impõe no máximo um texto por modelo no grupo."""


from collections.abc import Sequence

import numpy as np
from scipy.optimize import linear_sum_assignment


def _groups(groups: Sequence) -> dict:
    idx: dict = {}
    for i, g in enumerate(groups):
        idx.setdefault(g, []).append(i)
    return idx


def center_by_group(scores: np.ndarray, groups: Sequence) -> np.ndarray:
    out = scores.astype(float).copy()
    for rows in _groups(groups).values():
        out[rows] -= out[rows].mean(axis=0)
    return out


def assign_by_group(scores: np.ndarray, groups: Sequence) -> np.ndarray:
    """Devolve índices de classe. Grupos maiores que o nº de classes ficam com o argmax."""
    pred = scores.argmax(axis=1)
    for rows in _groups(groups).values():
        if len(rows) > scores.shape[1]:
            continue
        r, c = linear_sum_assignment(-scores[rows])
        pred[np.asarray(rows)[r]] = c
    return pred

In [ ]:
# ---- ntl/submission.py ----
"""Arquivo de submissão: as mesmas regras que o Kaggle usa para rejeitar."""


from collections.abc import Sequence
from pathlib import Path

import pandas as pd



def make_submission(ids: Sequence, preds: Sequence) -> pd.DataFrame:
    if len(ids) != len(preds):
        raise ValueError("ids e previsões com tamanhos diferentes")
    s = pd.DataFrame({"id": list(ids), "model": list(preds)})
    if s.isna().any().any():
        raise ValueError("valores nulos")
    if s["id"].duplicated().any():
        raise ValueError("id repetido")
    fora = set(s["model"]) - set(LABELS)
    if fora:
        raise ValueError(f"rótulos inválidos: {sorted(fora)}")
    return s


def write_submission(s: pd.DataFrame, out_dir: Path | str = ".") -> Path:
    p = Path(out_dir) / "submission.parquet"
    s.to_parquet(p, index=False)
    return p

In [ ]:
# Treina no train.csv inteiro, prevê o teste e aplica o pós-processamento da variante.
import numpy as np

train, test = load_train(), load_test()
model = four_view_lr()
model.fit(train["text"].to_numpy(), train["model"].to_numpy())
P = model.predict_proba(test["text"].fillna("").to_numpy())
classes = np.array(model.classes_)
L = np.log(np.clip(P, 1e-9, 1))
groups = capped_groups(test["text"].fillna("").tolist())
pred = classes[assign_by_group(L, groups)]
path = write_submission(make_submission(test["id"].tolist(), pred.tolist()))
print(path, len(test), "linhas")